# TASK 1: Retail Sales Discovery Engine
### Comprehensive Commercial Diagnostics, Profitability Profiling, and Discount Sensitivity Analysis

**Author**: Vijay Maheswari  
**Framework**: RISK Methodology (Reconnaissance, Intelligence, Solution, KPI Verification)  
**Dataset**: Retail Superstore Sales (9,994 validated records)


In [1]:
import sys
import os

# Ensure parent directory is in sys.path
for p in [os.path.abspath(".."), os.path.abspath(".")]:
    if p not in sys.path:
        sys.path.insert(0, p)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.loaders import load_superstore
from src.cleaning import clean_superstore
from src.metrics import calculate_retail_kpis
from src.validation import validate_superstore_schema

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = 'Arial'


## 1. Data Ingestion & Quality Audit
We ingest the raw dataset, perform null-value and duplicate checks, and apply deterministic cleaning rules while preserving raw provenance.


In [2]:
raw_df = load_superstore()
print(f"Raw dataset shape: {raw_df.shape}")

# Validate raw structure
audit_raw = validate_superstore_schema(raw_df)
print(audit_raw.summary())

# Clean dataset
df = clean_superstore(raw_df)
print(f"Cleaned dataset shape: {df.shape}")

# Validate cleaned structure
audit_clean = validate_superstore_schema(df)
print(audit_clean.summary())


Raw dataset shape: (10800, 21)
[PASSED] Superstore Sales: 10800 rows, 21 cols | Warnings: 1
Cleaned dataset shape: (9994, 31)
[PASSED] Superstore Sales: 9994 rows, 31 cols


## 2. Core Commercial Scorecard
We calculate the grand totals matching Power BI DAX definitions.


In [3]:
kpis = calculate_retail_kpis(df)
for k, v in kpis.items():
    if not isinstance(v, pd.DataFrame):
        print(f"{k}: {v:,}" if isinstance(v, (int, float)) else f"{k}: {v}")


Total Sales: 2,297,200.86
Total Profit: 286,397.02
Total Quantity: 37,873
Total Orders: 5,009
Total Customers: 793
Profit Margin %: 12.47
Average Order Value: 458.61
Negative Profit Order Lines: 1,871
Total Negative Profit Loss: -156,131.29


## 3. Visual Analytics & Diagnostic Charts
We examine monthly sales trajectories, category profitability, regional disparities, and discount elasticity.


In [4]:
from src.generate_retail_eda import generate_retail_visualizations

generate_retail_visualizations(df, "outputs/figures")
print("Visualizations rendered to outputs/figures/")


D:\intership\Datheon_Data_Analytics_&_Power_BI_Internship\vj\src\generate_retail_eda.py:54: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=reg, x="Region", y="Profit", palette="Blues_d")


Visualizations rendered to outputs/figures/


## 4. Discount Sensitivity Breakdown
Isolating the commercial impact of discounting reveals a severe margin collapse when discounts surpass 20%.


In [5]:
disc_perf = df.groupby("Discount Band", observed=False).agg(
    Order_Lines=("Order ID", "count"),
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum")
)
disc_perf["Profit_Margin_%"] = (disc_perf["Total_Profit"] / disc_perf["Total_Sales"]) * 100
disc_perf["Sales_Share_%"] = (disc_perf["Total_Sales"] / df["Sales"].sum()) * 100
disc_perf


,Order_Lines,Total_Sales,Total_Profit,Profit_Margin_%,Sales_Share_%
Discount Band,,,,,
None (0%),4798,1.087908e+06,320987.6032,29.505019,47.358004
Low (0-10%],94,5.436935e+04,9029.1770,16.607108,2.366765
Medium (10-20%],3709,7.921529e+05,91756.2975,11.583155,34.483397
High (>20%),1393,3.627701e+05,-135376.0560,-37.317309,15.791834


## 5. Sub-Category Value Destruction & Value Creation
Analyzing the bottom vs top subcategories isolates structural product line loss leaders.


In [6]:
subcat_diag = df.groupby(["Category", "Sub-Category"]).agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique"),
    Avg_Discount=("Discount", "mean")
).reset_index()
subcat_diag["Profit_Margin_%"] = (subcat_diag["Profit"] / subcat_diag["Sales"]) * 100
subcat_diag.sort_values("Profit", ascending=True)


,Category,Sub-Category,Sales,Profit,Orders,Avg_Discount,Profit_Margin_%
3,Furniture,Tables,206965.5320,-17725.4811,307,0.261285,-8.564460
0,Furniture,Bookcases,114879.9963,-3472.5560,224,0.211140,-3.022768
12,Office Supplies,Supplies,46673.5380,-1189.0995,187,0.076842,-2.547695
8,Office Supplies,Fasteners,3024.2800,949.5182,215,0.082028,31.396504
15,Technology,Machines,189238.6310,3384.7569,112,0.306087,1.788618
9,Office Supplies,Labels,12486.3120,5546.2540,346,0.068681,44.418672
5,Office Supplies,Art,27118.7920,6527.7870,731,0.074874,24.071083
7,Office Supplies,Envelopes,16476.4020,6964.1767,249,0.080315,42.267582
2,Furniture,Furnishings,91705.1640,13059.1436,877,0.138349,14.240358
4,Office Supplies,Appliances,107532.1610,18138.0054,451,0.166524,16.867517
